In [ ]:
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.mask import mask as rio_mask
from rasterio.features import shapes
from shapely.geometry import shape
from shapely.ops import unary_union
from shapely import make_valid
import folium
from folium import FeatureGroup, GeoJson
import json

In [ ]:
TIF_DIR  = r"D:\Arq-Azzoni\UrbanSprawl\Bases_dados\Vegetacao_mapbiomas"
LIMITE   = "./data/green_jundiai/limite_municipal_jundiai.shp"

ANOS = [
    (2000, "./data/soil_use_2000.shp"),
    (2010, "./data/soil_use_2010.shp"),
    (2023, "./data/soil_use_2023.shp"),
]

ESTILOS_URBANO = {
    2000: {"fillColor": "#d1d5db", "color": "#6b7280"},
    2010: {"fillColor": "#9ca3af", "color": "#4b5563"},
    2023: {"fillColor": "#6b7280", "color": "#374151"},
}

limite = gpd.read_file(LIMITE).to_crs(4326)
centro = limite.geometry.union_all().centroid

In [ ]:
def extrair_raster(ano: int, valores: list[int]) -> gpd.GeoDataFrame:
    """Recorta TIF ao limite de Jundiaí e vetoriza os pixels com valores dados."""
    tif = f"{TIF_DIR}/mapbiomas_urbano_sp_{ano}.tif"
    with rasterio.open(tif) as src:
        lim_proj = limite.to_crs(src.crs)
        geom_mask = [lim_proj.union_all().__geo_interface__]
        arr, transform = rio_mask(src, geom_mask, crop=True, filled=True, nodata=0)
        arr = arr[0].astype(np.uint16)
        crs = src.crs

    mask_bin = np.isin(arr, valores).astype(np.uint8)
    if mask_bin.sum() == 0:
        return gpd.GeoDataFrame(geometry=[], crs=crs).to_crs(4326)

    geoms = [shape(g) for g, _ in shapes(mask_bin, mask=mask_bin, transform=transform)]
    gdf = gpd.GeoDataFrame(geometry=geoms, crs=crs).to_crs(4326)
    # Dissolver para reduzir polígonos
    gdf = (
        gdf.to_crs(31983)
           .dissolve()
           .to_crs(4326)
           .explode(index_parts=False, ignore_index=True)
    )
    return gdf


def carregar_urbano(shp_path: str) -> gpd.GeoDataFrame:
    """Carrega polígono urbano do shapefile de uso do solo."""
    g = gpd.read_file(shp_path)
    g["geometry"] = g.geometry.apply(make_valid)
    g = g.to_crs(4326)
    g = g[g["soil_use"].astype(str).str.lower() == "urbano"].copy()
    return g

In [ ]:
# Vegetação = Formação Florestal (3) + Pastagem (15) + Mosaico de Usos (21) + Outras Lavouras Temp. (41)
VEG_VALORES = [3, 15, 21, 41]

dados = {}
for ano, shp in ANOS:
    print(f"Processando {ano}...")
    dados[ano] = {
        "urbano": carregar_urbano(shp),
        "veg":    extrair_raster(ano, VEG_VALORES),
    }
    # Verificar quais valores foram encontrados no clip de Jundiaí
    tif = f"{TIF_DIR}/mapbiomas_urbano_sp_{ano}.tif"
    with rasterio.open(tif) as src:
        lim_proj = limite.to_crs(src.crs)
        arr, _ = rio_mask(src, [lim_proj.union_all().__geo_interface__], crop=True, filled=True, nodata=0)
        vals, counts = np.unique(arr[0], return_counts=True)
        encontrados = {int(v): int(c) for v, c in zip(vals, counts) if int(v) in VEG_VALORES}
    print(f"  urbano: {len(dados[ano]['urbano'])} | veg polígonos: {len(dados[ano]['veg'])} | pixels por valor: {encontrados}")

print("\nConcluído.")

In [ ]:
m = folium.Map(location=[centro.y, centro.x], zoom_start=11, tiles=None)

folium.TileLayer("CartoDB positron", name="Carto Positron", overlay=False, control=True).add_to(m)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri", name="Satélite (Esri)", overlay=False, control=True
).add_to(m)

folium.GeoJson(
    json.loads(limite.to_json()),
    name="Limite Jundiaí", control=False,
    style_function=lambda f: {"color": "#FFD700", "weight": 2, "fillOpacity": 0}
).add_to(m)

# ── Manchas urbanas ──────────────────────────────────────────────────────────
for ano, shp in ANOS:
    d  = dados[ano]
    st = ESTILOS_URBANO[ano]
    if d["urbano"].empty:
        continue
    fg = FeatureGroup(name=f"Mancha Urbana {ano}", show=True)
    GeoJson(
        json.loads(d["urbano"].to_json()),
        name=None,
        style_function=lambda f, s=st: {
            "fillColor": s["fillColor"], "color": s["color"],
            "weight": 1, "fillOpacity": 0.45
        },
    ).add_to(fg)
    fg.add_to(m)

# ── Vegetação por ano ────────────────────────────────────────────────────────
CORES_VEG = {
    2000: {"fillColor": "#86efac", "color": "#166534"},
    2010: {"fillColor": "#4ade80", "color": "#15803d"},
    2023: {"fillColor": "#16a34a", "color": "#14532d"},
}

for ano, shp in ANOS:
    d  = dados[ano]
    cv = CORES_VEG[ano]
    if d["veg"].empty:
        continue
    fg = FeatureGroup(name=f"Vegetação {ano} (Mapbiomas)", show=True)
    GeoJson(
        json.loads(d["veg"].to_json()),
        name=None,
        style_function=lambda f, c=cv: {
            "fillColor": c["fillColor"], "color": c["color"],
            "weight": 0.5, "fillOpacity": 0.75
        },
    ).add_to(fg)
    fg.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m

In [ ]:
m.save("mapa_comparacao_urbana_vegetacao.html")
print("Salvo: mapa_comparacao_urbana_vegetacao.html")